In [1]:
# import libs
# if this throws an error, something went wrong installing dependencies or changing the kernel above!
import os
import torch
import torchaudio
import numpy as np
import random

In [2]:
!mfa model download dictionary english_us_arpa && \
mfa model download acoustic english_us_arpa

 WARNING  Local version of model already exists                                 
          (/home/sukiennik/Documents/MFA/pretrained_models/dictionary/english_us
          _arpa.dict). Use the --ignore_cache flag to force redownloading.      
 WARNING  Local version of model already exists                                 
          (/home/sukiennik/Documents/MFA/pretrained_models/acoustic/english_us_a
          rpa.zip). Use the --ignore_cache flag to force redownloading.         


In [3]:
# GPU Configuration
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

# Checking how many GPUs are available
n_gpus = torch.cuda.device_count()

if n_gpus > 1:
    os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
    print(f"🚀 Found {n_gpus} GPUs. Using both (0 and 1).")
elif n_gpus == 1:
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    print("🏠 Found 1 GPU. Using device 0.")
else:
    print("⚠️ No GPU found, using CPU (this will be slow).")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# System identity
os.environ["USER"] = "Sukiennik" 


# Set the path to the espeak library manually (Windows)
# os.environ['PHONEMIZER_ESPEAK_LIBRARY'] = r'C:\Program Files\eSpeak NG\libespeak-ng.dll'
# --- WSL UPDATE: Setting the path to the Linux espeak-ng library ---
# In Ubuntu/WSL, the library is a .so file, not a .dll
espeak_lib = "/usr/lib/x86_64-linux-gnu/libespeak-ng.so.1"
espeak_path = "/usr/bin/espeak-ng"

os.environ['PHONEMIZER_ESPEAK_PATH'] = espeak_path
os.environ['PHONEMIZER_ESPEAK_LIBRARY'] = espeak_lib

# Importing VoiceCraft modules
try:
    from data.tokenizer import AudioTokenizer, TextTokenizer
    from models import voicecraft
    print("✅ Libraries and VoiceCraft modules imported successfully!")
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("Tip: Make sure you are in the project root directory.")

🏠 Found 1 GPU. Using device 0.
✅ Libraries and VoiceCraft modules imported successfully!


/home/sukiennik/miniconda3/envs/voicecraft_linux/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
# # load model, encodec, and phn2num
# # # load model, tokenizer, and other necessary files
device = "cuda" if torch.cuda.is_available() else "cpu"
# from voiceCraft.models import voicecraft
# # reload voicecraft
# import importlib
# importlib.reload(voicecraft)
# from voiceCraft.models import voicecraft
voicecraft_name="gigaHalfLibri330M_TTSEnhanced_max16s.pth" # or giga330M.pth, giga830M.pth
ckpt_fn =f"./pretrained_models/{voicecraft_name}"
encodec_fn = "./pretrained_models/encodec_4cb2048_giga.th"
if not os.path.exists(ckpt_fn):
    os.system(f"wget https://huggingface.co/pyp1/VoiceCraft/resolve/main/{voicecraft_name}\?download\=true")
    os.system(f"mv {voicecraft_name}\?download\=true ./pretrained_models/{voicecraft_name}")
if not os.path.exists(encodec_fn):
    os.system(f"wget https://huggingface.co/pyp1/VoiceCraft/resolve/main/encodec_4cb2048_giga.th")
    os.system(f"mv encodec_4cb2048_giga.th ./pretrained_models/encodec_4cb2048_giga.th")

# load model, encodec, and phn2num
# # load model, tokenizer, and other necessary files
# device = "cuda" if torch.cuda.is_available() else "cpu"
# voicecraft_name="gigaHalfLibri330M_TTSEnhanced_max16s.pth" # or giga330M.pth, giga830M.pth
# ckpt_fn =f"./pretrained_models/{voicecraft_name}"
# encodec_fn = "./pretrained_models/encodec_4cb2048_giga.th"

ckpt = torch.load(ckpt_fn, map_location="cpu")
model = voicecraft.VoiceCraft(ckpt["config"])
model.load_state_dict(ckpt["model"])
model.to(device)
model.eval()
phn2num = ckpt['phn2num']
text_tokenizer = TextTokenizer(backend="es2peak")
audio_tokenizer = AudioTokenizer(signature=encodec_fn, device=device) # will also put the neural codec model on gpu

/home/sukiennik/miniconda3/envs/voicecraft_linux/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:30: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")


In [13]:
if torch.cuda.is_available():
    print(f"VRAM Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"VRAM Reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

VRAM Allocated: 1.53 GB
VRAM Reserved: 1.89 GB


In [14]:
# Convert model to half precision 
model.half() 

# Alternatively, if you haven't moved to device yet:
# model.to(device, dtype=torch.float16)

print("✨ Model converted to float16 (Half Precision)")

✨ Model converted to float16 (Half Precision)


In [15]:
if torch.cuda.is_available():
    print(f"VRAM Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"VRAM Reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

VRAM Allocated: 0.89 GB
VRAM Reserved: 1.89 GB


In [35]:
# Prepare your audio# point to the original audio whose speech you want to clone
# write down the transcript for the file, or run whisper to get the transcript (and you can modify it if it's not accurate), save it as a .txt file
# orig_audio = "./demo/84_121550_000074_000000.wav"
# orig_transcript = "But when I had approached so near to them The common object, which the sense deceives, Lost not by distance any of its marks,"
orig_audio = "./demo/Daniella.wav"
orig_transcript = "Hello, my name is Daniella. I am currently exploring new technologies and testing how this voice model works. It is really exciting to see what artificial intelligence can do today!"
temp_folder = "./demo/temp"
os.makedirs(temp_folder, exist_ok=True)
os.system(f"cp {orig_audio} {temp_folder}")
filename = os.path.splitext(orig_audio.split("/")[-1])[0]
with open(f"{temp_folder}/{filename}.txt", "w") as f:
    f.write(orig_transcript)
# run MFA to get the alignment
align_temp = f"{temp_folder}/mfa_alignments"
os.system(f"mfa align -j 1 --clean --output_format csv {temp_folder} english_us_arpa english_us_arpa {align_temp}")
# # if the above fails, it could be because the audio is too hard for the alignment model, increasing the beam size usually solves the issue
# os.system(f"mfa align -j 1 --clean --output_format csv {temp_folder} english_us_arpa english_us_arpa {align_temp} --beam 1000 --retry_beam 2000")

 INFO     Setting up corpus information...                                      
 INFO     Loading corpus from source files...                                   


   2% ╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/100  [ 0:00:01 < -:--:-- , ? it/s ]
   0% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/2  [ 0:00:00 < -:--:-- , ? it/s ]

 INFO     Found 1 speaker across 2 files, average number of utterances per      
          speaker: 2.0                                                          
 INFO     Initializing multiprocessing jobs...                                  
 INFO     Normalizing text...                                                   


 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2  [ 0:00:02 < 0:00:00 , 1 it/s ] ? it/s ]


 INFO     Creating corpus split for feature generation...                       


...

 INFO     Collecting phone and word alignments from alignment lattices...       


 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2  [ 0:00:02 < 0:00:00 , 1 it/s ] ? it/s ]


 WARNING  Alignment analysis not available without using postgresql             
 INFO     Exporting alignment TextGrids to demo/temp/mfa_alignments...          
 INFO     Finished exporting TextGrids to demo/temp/mfa_alignments!             
 INFO     Done! Everything took 34.624 seconds                                  


 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2  [ 0:00:00 < 0:00:00 , ? it/s ]


0

In [43]:
# take a look the csv file in VoiceCraft/demo/temp/mfa_alignment, decide which part of the audio to use as prompt
cut_off_sec = 3.01 # NOTE: according to forced-alignment file demo/temp/mfa_alignments/84_121550_000074_000000.csv, the word "common" stop as 3.01 sec, this should be different for different audio
target_transcript = "But when I had approached so near to them The common I cannot believe that the same model can also do text to speech synthesis as well!"
target_transcript = "But when I had approached so near to them The common Daniella works as sofware engineer at Texas Instruments"

cut_off_sec = 6.01 # NOTE: according to forced-alignment file demo/temp/mfa_alignments/84_121550_000074_000000.csv, the word "common" stop as 3.01 sec, this should be different for different audio
target_transcript = "Hello, my name is Daniella. I am currently exploring new technologies and testing how amazing this model is, i can't beleive it works"
target_transcript = "Hello, my name is Daniella. I am currently exploring new technologies and testing how amazing this model is, Ah-nee oh-heh-vet AI" #Ani ohevet AI"

# NOTE: 3 sec of reference is generally enough for high quality voice cloning, but longer is generally better, try e.g. 3~6 sec.
audio_fn = f"{temp_folder}/{filename}.wav"
info = torchaudio.info(audio_fn)
audio_dur = info.num_frames / info.sample_rate

assert cut_off_sec < audio_dur, f"cut_off_sec {cut_off_sec} is larger than the audio duration {audio_dur}"
prompt_end_frame = int(cut_off_sec * info.sample_rate)

# run the model to get the output
# hyperparameters for inference
codec_audio_sr = 16000
codec_sr = 50
top_k = 0
top_p = 0.8
temperature = 1
silence_tokens=[1388,1898,131]
kvcache = 1 # NOTE if OOM, change this to 0, or try the 330M model

# NOTE adjust the below three arguments if the generation is not as good
stop_repetition = 3 # NOTE if the model generate long silence, reduce the stop_repetition to 3, 2 or even 1
sample_batch_size = 3 # for gigaHalfLibri330M_TTSEnhanced_max16s.pth, 1 or 2 should be fine since the model is trained to do TTS, for the other two models, might need a higher number. NOTE: if the if there are long silence or unnaturally strecthed words, increase sample_batch_size to 5 or higher. What this will do to the model is that the model will run sample_batch_size examples of the same audio, and pick the one that's the shortest. So if the speech rate of the generated is too fast change it to a smaller number.
seed = 1 # change seed if you are still unhappy with the result

def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
seed_everything(seed)

decode_config = {
    'top_k': top_k,
    'top_p': top_p,
    'temperature': temperature,
    'stop_repetition': stop_repetition,
    'kvcache': kvcache, 
    "codec_audio_sr": codec_audio_sr,
    "codec_sr": codec_sr, 
    "silence_tokens": silence_tokens, 
    "sample_batch_size": sample_batch_size}

from inference_tts_scale import inference_one_sample
with torch.no_grad():
    # This context manager handles the float16/float32 mixing
    with torch.cuda.amp.autocast():
        concated_audio, gen_audio = inference_one_sample(
            model, 
            ckpt["config"], 
            phn2num, 
            text_tokenizer, 
            audio_tokenizer, 
            audio_fn, 
            target_transcript, 
            device, 
            decode_config, 
            prompt_end_frame
        )

# save segments for comparison
concated_audio, gen_audio = concated_audio[0].cpu(), gen_audio[0].cpu()
# logging.info(f"length of the resynthesize orig audio: {orig_audio.shape}")


# display the audio
from IPython.display import Audio
print("concatenate prompt and generated:")
display(Audio(concated_audio, rate=codec_audio_sr))

print("generated:")
display(Audio(gen_audio, rate=codec_audio_sr))

# # save the audio
# # output_dir
# output_dir = "/home/sukiennik/projects/VoiceCraft/demo/generated_tts"
# os.makedirs(output_dir, exist_ok=True)
# seg_save_fn_gen = f"{output_dir}/{os.path.basename(audio_fn)[:-4]}_gen_seed{seed}.wav"
# seg_save_fn_concat = f"{output_dir}/{os.path.basename(audio_fn)[:-4]}_concat_seed{seed}.wav"

# torchaudio.save(seg_save_fn_gen, gen_audio.float(), codec_audio_sr) # add .float() to convert to 32 bit back
# torchaudio.save(seg_save_fn_concat, concated_audio.float(), codec_audio_sr) # add .float() to convert to 32 bit back

# if you get error importing T5 in transformers
# try
# pip uninstall Pillow
# pip install Pillow
# you are might get warnings like WARNING:phonemizer:words count mismatch on 300.0% of the lines (3/1), this can be safely ignored

concatenate prompt and generated:


generated:


In [44]:
# save the audio
# output_dir
output_dir = "/home/sukiennik/projects/VoiceCraft/demo/generated_tts"
os.makedirs(output_dir, exist_ok=True)
seg_save_fn_gen = f"{output_dir}/{os.path.basename(audio_fn)[:-4]}_gen_seed{seed}_daniella.wav" 
seg_save_fn_concat = f"{output_dir}/{os.path.basename(audio_fn)[:-4]}_concat_seed{seed}_daniella.wav"

torchaudio.save(seg_save_fn_gen, gen_audio.float(), codec_audio_sr) # add .float() to convert to 32 bit back
torchaudio.save(seg_save_fn_concat, concated_audio.float(), codec_audio_sr) # add .float() to convert to 32 bit back

In [8]:
# Testing if the tokenizer can convert Hebrew to tokens (English comment)
# text_input = "שלום"
text_input = "shalom"
try:
    # In VoiceCraft, we usually use encode or call
    encoded_text = text_tokenizer.to_vector(text_input) 
    print(f"Text: {text_input}")
    print(f"Tokens: {encoded_text}")
except Exception as e:
    # If it fails, let's try to see what it DOES expect
    print(f"Failed with Hebrew characters. Error: {e}")
    
    # Try with English transliteration as a fallback
    test_english = "shalom"
    encoded_english = text_tokenizer.to_vector(test_english)
    print(f"\nFallback Text: {test_english}")
    print(f"Fallback Tokens: {encoded_english}")

Failed with Hebrew characters. Error: 'TextTokenizer' object has no attribute 'to_vector'


AttributeError: 'TextTokenizer' object has no attribute 'to_vector'

In [9]:
# Check available methods (English comment)
print(dir(text_tokenizer))

# Check what happens if we just call it (English comment)
try:
    test_res = text_tokenizer("hello")
    print(f"Direct call result: {test_res}")
except Exception as e:
    print(f"Direct call failed: {e}")

['__call__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'backend', 'separator', 'to_list']
Direct call result: [['h', 'ə', 'l', 'oʊ']]


In [10]:
# Testing with direct call (English comments)
def test_input(txt):
    try:
        # Most tokenizers use __call__ or .encode
        res = text_tokenizer(txt)
        print(f"SUCCESS | Input: {txt} | Output: {res}")
    except Exception as e:
        print(f"FAILED  | Input: {txt} | Error: {e}")

print("--- Testing Tokenizer ---")
test_input("hello")
test_input("shalom")
test_input("שלום")

--- Testing Tokenizer ---
SUCCESS | Input: hello | Output: [['h', 'ə', 'l', 'oʊ']]
SUCCESS | Input: shalom | Output: [['ʃ', 'æ', 'l', 'ə', 'm']]
SUCCESS | Input: שלום | Output: [['h', 'iː', 'b', 'ɹ', 'uː', 'ʃ', 'i', 'n', '_', 'h', 'iː', 'b', 'ɹ', 'uː', 'l', 'æ', 'm', 'e', 'd', '_', 'h', 'iː', 'b', 'ɹ', 'uː', 'v', 'æ', 'v', '_', 'h', 'iː', 'b', 'ɹ', 'uː', 'm', 'e', 'm']]
